# Document Reranking with Jina Reranker v3

This notebook demonstrates how to use the `jinaai/jina-reranker-v3` model with CrossEncoder for document reranking.

In [ ]:
# Install required packages (uncomment if needed)
# !pip install sentence-transformers torch

In [1]:
# sys.path.insert(0, str(Path(__file__).parent.parent / "experiments"))
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "scripts"))
sys.path.insert(0, str(Path.cwd().parent))
import torch
import numpy as np
from typing import List, Tuple
from scripts.rerank_with_head_weights import load_reranker_model


## Load the CrossEncoder Model

In [2]:
# Load Jina Reranker v3 model
# model = CrossEncoder('jinaai/jina-reranker-v3', device='cuda' if torch.cuda.is_available() else 'cpu')
jina_reranker_v3_model_path = 'jinaai/jina-reranker-v3'
granite_reranker_model_path = 'ibm-granite/granite-embedding-reranker-english-r2'
model, model_type = load_reranker_model(granite_reranker_model_path,
                                        device='cuda' if torch.cuda.is_available() else 'cpu',
                                        model_kwargs = {'dtype': torch.bfloat16, 'attn_implementation': "flash_attention_2"})
print(f"Model loaded on: {model.device}")

Loading cross-encoder model: ibm-granite/granite-embedding-reranker-english-r2
Device: cuda
Detected reranker type: cross-encoder
Model loaded on: cuda


from transformers import AutoModel
model = AutoModel.from_pretrained(jina_reranker_v3_model_path, dtype="auto", trust_remote_code=True)
model.eval()

In [3]:
query = "What are the health benefits of green tea?"
documents = [
    "Green tea contains antioxidants called catechins that may help reduce inflammation and protect cells from damage.",
    "El precio del café ha aumentado un 20% este año debido a problemas en la cadena de suministro.",
    "Studies show that drinking green tea regularly can improve brain function and boost metabolism.",
    "Basketball is one of the most popular sports in the United States.",
    "绿茶富含儿茶素等抗氧化剂，可以降低心脏病风险，还有助于控制体重。",
    "Le thé vert est riche en antioxydants et peut améliorer la fonction cérébrale.",
]

# Rerank documents
results = model.predict([[query, d] for d in documents])

# Results are sorted by relevance score (highest first)
for i, result in enumerate(results):
    print(f"Score: {result:.4f}")
    print(f"Document: {documents[i][:100]}...")
    print()

InductorError: PermissionError: [Errno 13] Permission denied: 'nvcc'

Set TORCHDYNAMO_VERBOSE=1 for the internal stack trace (please do this especially if you're reporting a bug to PyTorch). For even more developer context, set TORCH_LOGS="+dynamo"


In [4]:
# Sample query
query = "What is machine learning?"

# Sample documents to rerank
documents = [
    "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.",
    "The weather today is sunny with a high of 75 degrees Fahrenheit.",
    "Deep learning is a type of machine learning based on artificial neural networks with multiple layers.",
    "Python is a popular programming language used for web development, data science, and automation.",
    "Supervised learning is a machine learning approach where models are trained on labeled data.",
    "The capital of France is Paris, known for the Eiffel Tower and its rich history.",
    "Neural networks are computing systems inspired by biological neural networks in animal brains.",
    "Basketball is a team sport played on a rectangular court with two hoops."
]

print(f"Query: {query}")
print(f"\nNumber of documents: {len(documents)}")

Query: What is machine learning?

Number of documents: 8


In [5]:

query_doc_pairs = [[query, doc] for doc in documents]

# Get relevance scores
scores = model.predict(query_doc_pairs)

print("Relevance scores:")
for i, score in enumerate(scores):
    print(f"Doc {i}: {score:.4f}")

InductorError: PermissionError: [Errno 13] Permission denied: 'nvcc'

Set TORCHDYNAMO_VERBOSE=1 for the internal stack trace (please do this especially if you're reporting a bug to PyTorch). For even more developer context, set TORCH_LOGS="+dynamo"


## Sort and Display Reranked Results

In [6]:
# Sort documents by score (descending)
doc_score_pairs = list(zip(documents, scores, range(len(documents))))
doc_score_pairs.sort(key=lambda x: x[1], reverse=True)

print("=" * 80)
print("RERANKED RESULTS")
print("=" * 80)
print(f"\nQuery: {query}\n")

for rank, (doc, score, original_idx) in enumerate(doc_score_pairs, 1):
    print(f"Rank {rank} (Original position: {original_idx})")
    print(f"Score: {score:.4f}")
    print(f"Document: {doc}")
    print("-" * 80)

NameError: name 'scores' is not defined

## Batch Processing Example

In [ ]:
# Example with multiple queries
queries = [
    "What is machine learning?",
    "Tell me about Paris",
    "How does basketball work?"
]

print("Batch reranking for multiple queries:\n")

for query_idx, query in enumerate(queries, 1):
    print(f"\n{'='*60}")
    print(f"Query {query_idx}: {query}")
    print('='*60)
    
    # Create pairs and get scores
    pairs = [[query, doc] for doc in documents]
    scores = model.predict(pairs)
    
    # Get top 3 results
    top_indices = np.argsort(scores)[::-1][:3]
    
    for rank, idx in enumerate(top_indices, 1):
        print(f"\n  {rank}. Score: {scores[idx]:.4f}")
        print(f"     {documents[idx][:100]}..." if len(documents[idx]) > 100 else f"     {documents[idx]}")

## Performance Metrics

In [ ]:
# Simple relevance check (manual labels for demo)
# In practice, you would have ground truth relevance labels
relevant_doc_indices = {0, 2, 4, 6}  # Indices of ML-related documents

# Calculate precision@k
def precision_at_k(ranked_indices: List[int], relevant_set: set, k: int) -> float:
    top_k = ranked_indices[:k]
    relevant_in_top_k = sum(1 for idx in top_k if idx in relevant_set)
    return relevant_in_top_k / k

# Get ranked indices for our first query
query = "What is machine learning?"
pairs = [[query, doc] for doc in documents]
scores = model.predict(pairs)
ranked_indices = np.argsort(scores)[::-1].tolist()

print("Relevance Metrics:")
for k in [1, 3, 5]:
    p_at_k = precision_at_k(ranked_indices, relevant_doc_indices, k)
    print(f"Precision@{k}: {p_at_k:.3f}")